# Flocking Behavior Simulation

This notebook is the narrative companion to `flocking_simulation.py` (2D
Reynolds boids on AMBER's OOP lane). For a dense NumPy kernel see
`flocking_tensor.py`.

Agent state lives in the live `agents_df` table. Do **not** concatenate
per-step snapshots into that frame — history belongs in `record_model` or
a separate list.

## About the model

Craig Reynolds' boids model (1986) uses three steering behaviors:

- **Separation**: steer to avoid crowding local flockmates
- **Alignment**: steer towards the average heading of local flockmates
- **Cohesion**: steer to move toward the average position of local flockmates

This notebook is a simplified 2D implementation (plus a border-avoidance
rule). Neighbour search is O(N²); keep the population modest.

**Requirements:** `pip install ambr`. Plots need `pip install 'ambr[viz]'`.


In [ ]:
import ambr as am
import numpy as np

try:
    import matplotlib.pyplot as plt
except Exception as exc:
    plt = None
    print(f"matplotlib unavailable — plots will be skipped ({exc!r})")
    print("Install with: pip install 'ambr[viz]'")


def _normalize(v):
    n = float(np.linalg.norm(v))
    return v if n == 0.0 else v / n


## Model definition

Each `Boid` reads neighbour positions from the columnar view
(`model.agents.numpy(...)`) and writes `x`, `y`, `vx`, `vy` back through
the OOP attribute path. The model records mean speed each step.


In [ ]:
class Boid(am.Agent):
    def setup(self):
        size = float(self.model.p.get("size", 40))
        self.x = float(self.model.rng.random() * size)
        self.y = float(self.model.rng.random() * size)
        ang = float(self.model.rng.random() * 2 * np.pi)
        self.vx = float(np.cos(ang))
        self.vy = float(np.sin(ang))

    def flock(self):
        m = self.model
        p = m.p
        positions = np.column_stack([m.agents.numpy("x"), m.agents.numpy("y")])
        velocities = np.column_stack(
            [m.agents.numpy("vx"), m.agents.numpy("vy")]
        )
        me = np.array([self.x, self.y], dtype=float)
        d = np.linalg.norm(positions - me, axis=1)
        d[int(self.id)] = np.inf

        outer = d <= float(p.get("outer_radius", 8.0))
        inner = d <= float(p.get("inner_radius", 2.5))

        v = np.zeros(2, dtype=float)
        if outer.any():
            center = positions[outer].mean(axis=0)
            v += (center - me) * float(p.get("cohesion_strength", 0.01))
            avg_v = velocities[outer].mean(axis=0)
            v += (avg_v - np.array([self.vx, self.vy])) * float(
                p.get("alignment_strength", 0.2)
            )
        if inner.any():
            away = me - positions[inner]
            v += away.sum(axis=0) * float(p.get("separation_strength", 0.05))

        size = float(p.get("size", 40))
        border = float(p.get("border_distance", 5.0))
        strength = float(p.get("border_strength", 0.3))
        if self.x < border:
            v[0] += strength
        elif self.x > size - border:
            v[0] -= strength
        if self.y < border:
            v[1] += strength
        elif self.y > size - border:
            v[1] -= strength

        vel = _normalize(np.array([self.vx, self.vy]) + v)
        self.vx, self.vy = float(vel[0]), float(vel[1])
        self.x = float(np.clip(self.x + self.vx, 0.0, size))
        self.y = float(np.clip(self.y + self.vy, 0.0, size))


class BoidsModel(am.Model):
    def setup(self):
        n = int(self.p.get("population", 40))
        self.agents = am.AgentList(self, n, Boid)

    def step(self):
        self.agents.flock()

    def update(self):
        vx = self.agents.numpy("vx")
        vy = self.agents.numpy("vy")
        speed = np.sqrt(vx * vx + vy * vy).mean()
        self.record_model("mean_speed", float(speed))


## Single run

A 40-boid, 30-step run is enough to see headings align. Final positions
come from `results["agents"]` (end-of-run state, not a history stack).


In [ ]:
params = {
    "population": 40,
    "size": 40,
    "steps": 30,
    "seed": 0,
    "show_progress": False,
    "inner_radius": 2.5,
    "outer_radius": 8.0,
    "cohesion_strength": 0.01,
    "separation_strength": 0.05,
    "alignment_strength": 0.2,
    "border_distance": 5.0,
    "border_strength": 0.3,
}
model = BoidsModel(params)
results = model.run()

print("Smoke run OK:", results["info"]["status"])
print("metrics:", results["model"].columns)
print(results["model"].tail(3).to_dicts())

agents = results["agents"]
print("final agents:", agents.height, "cols=", agents.columns)

if plt is not None:
    fig, ax = plt.subplots(figsize=(5, 5))
    ax.quiver(
        agents["x"].to_numpy(),
        agents["y"].to_numpy(),
        agents["vx"].to_numpy(),
        agents["vy"].to_numpy(),
        angles="xy",
        scale_units="xy",
        scale=0.5,
    )
    ax.set_title("Boids final positions / headings")
    ax.set_xlabel("x")
    ax.set_ylabel("y")
    fig.tight_layout()
